# 01 · Data Loading & Cleaning
Phase 1: Load all raw data, normalize team names, validate, build team registry.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from src.data_loader import load_all
np.random.seed(42)


## 1.1 Load All Datasets

In [ ]:
data = load_all(verbose=True)
results    = data['results_primary']
results_full = data['results_full']
fixtures   = data['group_fixtures']
ko_slots   = data['knockout_slots']
rankings   = data['fifa_rankings']
shootouts  = data['shootouts']
wc_stats   = data['wc_stats']
registry   = data['team_registry']
former     = data['former_names']
print("\nTeam registry preview:")
registry.head(10)


## 1.2 Validate Results Dataset

In [ ]:
print("Results shape:", results.shape)
print("Date range:", results['date'].min().date(), "→", results['date'].max().date())
print("Unique home teams:", results['home_team'].nunique())
print("Unique away teams:", results['away_team'].nunique())
print("\nMissing values:")
print(results.isnull().sum()[results.isnull().sum() > 0])
print("\nTournament types (top 15):")
print(results['tournament'].value_counts().head(15))


## 1.3 Group Fixtures Validation

In [ ]:
print("Group fixtures shape:", fixtures.shape)
print("\nGroups and team counts:")
print(fixtures.groupby('group')[['home_team','away_team']].nunique())
print("\nAll WC 2026 teams:")
all_teams = sorted(set(fixtures['home_team']) | set(fixtures['away_team']))
print(f"  {len(all_teams)} teams: {all_teams}")


## 1.4 Team Registry Summary

In [ ]:
print("Registry shape:", registry.shape)
print("\nBy confederation:")
print(registry.groupby('confederation').agg(
    count=('team_name','count'),
    avg_rank=('fifa_ranking','mean'),
    avg_points=('fifa_points','mean')
).round(1))

print("\nHost nations:")
print(registry[registry['is_host']])

print("\nFIFA Rankings — Top 10 WC teams:")
print(registry.nsmallest(10,'fifa_ranking')[['team_name','group','confederation','fifa_ranking','fifa_points']])


## 1.5 Shootouts Validation

In [ ]:
print("Shootouts shape:", shootouts.shape)
print("Date range:", shootouts['date'].min().date(), "→", shootouts['date'].max().date())
print("\nSample:")
shootouts.head(5)


## 1.6 WC Historical Stats Preview

In [ ]:
if wc_stats is not None:
    print("WC historical stats shape:", wc_stats.shape)
    print("Columns:", wc_stats.columns.tolist()[:15])
    print("Years covered:", wc_stats['Year'].min(), "→", wc_stats['Year'].max())
    print("\nCards distribution:")
    if 'home_yellow_card_long' in wc_stats.columns:
        print("  Yellow card data available")
    if 'home_red_card' in wc_stats.columns:
        rc_h = wc_stats['home_red_card'].fillna('').astype(str).str.count('·')
        rc_a = wc_stats['away_red_card'].fillna('').astype(str).str.count('·')
        print(f"  Red cards per match: { (rc_h.sum() + rc_a.sum()) / len(wc_stats) :.3f}")
else:
    print("WC historical stats not available — using empirical priors")


## ✅ Phase 1 Complete
All data loaded and validated. Processed files saved to `data/processed/`.